In [1]:
#RFSoC_test_script.py created 2025-09-26 12:29:55.395467

import json
import numpy as np
import urllib.request
HOST = "pynq"
# HOST = "129.97.41.202"
PORT = 9009
URL  = f"http://{HOST}:{PORT}/upload_rows"
 
def upload_rows(rows, time_unit="us"):
    payload = {"rows": rows, "time_unit": time_unit}
    data = json.dumps(payload).encode()
    req = urllib.request.Request(URL, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())
 
micro_freq = 455 #+20.767

rows = [
    [1, [micro_freq], [0], [1], [0], 0],
    [2, [500], [0], [1], [0], 0]
]



resp = upload_rows(rows, time_unit="us")
print(json.dumps(resp, indent=2))

{
  "status": "ok",
  "descriptors": 5
}


In [2]:
import json
import urllib.request
import urllib.error


# RFSoC network address
RFSOC_IP = "192.168.168.43"
RFSOC_PORT = 9010

BASE_URL = f"http://{RFSOC_IP}:{RFSOC_PORT}"


def check_rfsoc_server(timeout=5):
    """Check whether the RFSoC server is reachable."""
    try:
        with urllib.request.urlopen(
            f"{BASE_URL}/health",
            timeout=timeout,
        ) as response:
            result = json.loads(
                response.read().decode("utf-8")
            )

        print("RFSoC server is reachable:")
        print(json.dumps(result, indent=2))

        return result

    except Exception as error:
        raise RuntimeError(
            f"Could not reach {BASE_URL}: {error}"
        ) from error


def upload_rows(
    rows,
    dac="DAC0",
    time_unit="us",
    timeout=30,
):
    """
    Upload a row table to DAC0 or DAC2.

    dac:
        "DAC0" or "DAC2"

    time_unit:
        "ns", "us", "ms", or "s"
    """
    dac = str(dac).upper()

    if dac not in {"DAC0", "DAC2"}:
        raise ValueError(
            "dac must be 'DAC0' or 'DAC2'."
        )

    if time_unit not in {"ns", "us", "ms", "s"}:
        raise ValueError(
            "time_unit must be 'ns', 'us', 'ms', or 's'."
        )

    payload = {
        "dac": dac,
        "rows": rows,
        "time_unit": time_unit,
    }

    request = urllib.request.Request(
        f"{BASE_URL}/upload_rows",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=timeout,
        ) as response:
            result = json.loads(
                response.read().decode("utf-8")
            )

    except urllib.error.HTTPError as error:
        response_text = error.read().decode(
            "utf-8",
            errors="replace",
        )

        raise RuntimeError(
            f"RFSoC returned HTTP {error.code}: "
            f"{response_text}"
        ) from error

    except urllib.error.URLError as error:
        raise RuntimeError(
            f"Could not connect to {BASE_URL}: "
            f"{error.reason}"
        ) from error

    return result


def upload_dac0(rows, time_unit="us"):
    return upload_rows(
        rows=rows,
        dac="DAC0",
        time_unit=time_unit,
    )


def upload_dac2(rows, time_unit="us"):
    return upload_rows(
        rows=rows,
        dac="DAC2",
        time_unit=time_unit,
    )


# Check the connection.
check_rfsoc_server()

RFSoC server is reachable:
{
  "ok": true,
  "service": "RFSoC DAC table upload server",
  "uptime_s": 443444.225,
  "overlay": "/home/xilinx/jupyter_notebooks/Server_for_control/Fast_TTL_fix_V5.xsa",
  "bind": "0.0.0.0:9010",
  "access_urls": [
    "http://192.168.168.43:9010",
    "http://192.168.2.99:9010",
    "http://192.168.3.1:9010"
  ],
  "dac_ports": {
    "DAC0": {
      "dma_name": "axi_dma_dac",
      "axis_input": "s00_axis",
      "rfdc_output": "vout00"
    },
    "DAC2": {
      "dma_name": "axi_dma_dac_b",
      "axis_input": "s20_axis",
      "rfdc_output": "vout20"
    }
  },
  "last_upload": {
    "ok": true,
    "dac": "DAC0",
    "dma": "axi_dma_dac",
    "axis_input": "s00_axis",
    "rfdc_output": "vout00",
    "rows": 1,
    "active_tone_lanes": 2,
    "segments": 2,
    "descriptors": 5,
    "bytes_uploaded": 160,
    "input_time_unit": "us",
    "build_ms": 1.459,
    "dma_ms": 0.218,
    "uploaded_unix_time": 1787111455.6821938
  }
}


{'ok': True,
 'service': 'RFSoC DAC table upload server',
 'uptime_s': 443444.225,
 'overlay': '/home/xilinx/jupyter_notebooks/Server_for_control/Fast_TTL_fix_V5.xsa',
 'bind': '0.0.0.0:9010',
 'access_urls': ['http://192.168.168.43:9010',
  'http://192.168.2.99:9010',
  'http://192.168.3.1:9010'],
 'dac_ports': {'DAC0': {'dma_name': 'axi_dma_dac',
   'axis_input': 's00_axis',
   'rfdc_output': 'vout00'},
  'DAC2': {'dma_name': 'axi_dma_dac_b',
   'axis_input': 's20_axis',
   'rfdc_output': 'vout20'}},
 'last_upload': {'ok': True,
  'dac': 'DAC0',
  'dma': 'axi_dma_dac',
  'axis_input': 's00_axis',
  'rfdc_output': 'vout00',
  'rows': 1,
  'active_tone_lanes': 2,
  'segments': 2,
  'descriptors': 5,
  'bytes_uploaded': 160,
  'input_time_unit': 'us',
  'build_ms': 1.459,
  'dma_ms': 0.218,
  'uploaded_unix_time': 1787111455.6821938}}

In [16]:
TABLE_DAC0 = [
    [0, False, [
        [0, [
            # f_MHz, phase_rad, amp, duration_us, phase_mode,
            # rep_rate_mode, enable, phase_latency_cycles, label
            [150.000, 0.0, 0.5, 5e6,
             0, 0, True, 0, "DAC0 tone 0"],
        ]],
        [1, [
            [150.000 + 2.947, 0, 0.5, 5e6,
             0, -1, True, 0, "DAC0 tone 1, rep +1"],
        ]],
    ]],
]

response = upload_dac0(
    TABLE_DAC0,
    time_unit="us",
)

print(json.dumps(response, indent=2))

{
  "ok": true,
  "dac": "DAC0",
  "dma": "axi_dma_dac",
  "axis_input": "s00_axis",
  "rfdc_output": "vout00",
  "rows": 1,
  "active_tone_lanes": 2,
  "segments": 2,
  "descriptors": 5,
  "bytes_uploaded": 160,
  "input_time_unit": "us",
  "build_ms": 1.914,
  "dma_ms": 0.218,
  "uploaded_unix_time": 1784233416.0756922
}


In [4]:
import numpy as np
TABLE_DAC0 = [
    [0, False, [
        [0, [
            # f_MHz, phase_rad, amp, duration_us, phase_mode,
            # rep_rate_mode, enable, phase_latency_cycles, label
            [150.000, 0.0, 0.5, 5e6,
             0, 0, True, 0, "DAC0 tone 0"],
        ]],
        [1, [
            [150.000 + 2.947, 0, 0.5, 5e6,
             0, -1, True, 0, "DAC0 tone 1, rep +1"],
        ]],
    ]],
]
RamanPulseTime = 0.2
tone_1 = 20
tone_2 = 20
rep_rate_stabilisation = 106
wait_time = 0.23456

TABLE_DAC0 = [
    [0, False, [
        [0, [
            # f_MHz, phase_rad, amp, duration_us, phase_mode,
            # rep_rate_mode, enable, phase_latency_cycles, label
            [0, 0.0, 1.0, 5e6,
                0, 0, False, 0, "Dummy Tone"],
        ]],
    ]],
    [1, False, [
        [0, [                    # Tone channel 0
            # freq, phase, amp, duration_us, phase_mode,
            # rep_rate_mode, enable, latency, label

            [tone_1, 0.0, 0.5, RamanPulseTime/2,
             1, 0, True, 0, "U1"],

            [tone_1, 0.0, 0.00, wait_time,
             1, 0, True, 0, "RF OFF"],

            [tone_1, np.pi, 0.5, RamanPulseTime/2,
             1, 0, True, 0, "U2"],
        ]],
        [1, [                    # Tone channel 1
            # freq, phase, amp, duration_us, phase_mode,
            # rep_rate_mode, enable, latency, label

            [tone_2, 0.0, 0.5, RamanPulseTime/2,
             1, 0, True, 0, "U1"],

            [tone_2, 0.0, 0.00, wait_time,
             1, 0, True, 0, "RF OFF"],

            [tone_2, np.pi, 0.5, RamanPulseTime/2,
             1, 0, True, 0, "U2"],
        ]],
    ]],
    
]

tone_1 = 150
tone_2 = 150 + 12.345678

TABLE_DAC0 = [
    [0, False, [
        [0, [
            # f_MHz, phase_rad, amp, duration_us, phase_mode,
            # rep_rate_mode, enable, phase_latency_cycles, label
            [tone_1, 0.0, 0.5/2, 5e6,
             0, 0, True, 0, "DAC0 tone 0"],
        ]],
        [1, [
            [tone_2, 0, 0.5, 5e6,
             0, 0.0, True, 0, "DAC0 tone 1, rep -1"],
        ]],
    ]],
]

upload_dac0(TABLE_DAC0,time_unit="us")
# upload_dac2(
#     TABLE_DAC0,
#     time_unit="us",
#     )

{'ok': True,
 'dac': 'DAC0',
 'dma': 'axi_dma_dac',
 'axis_input': 's00_axis',
 'rfdc_output': 'vout00',
 'rows': 1,
 'active_tone_lanes': 2,
 'segments': 2,
 'descriptors': 5,
 'bytes_uploaded': 160,
 'input_time_unit': 'us',
 'build_ms': 1.488,
 'dma_ms': 0.22,
 'uploaded_unix_time': 1787192318.8856087}